# Paralelizace

Existují dva hlavní způsoby, jak paralelizovat výpočet:

- Použít více počítačů (nebo lépe více CPU).

- Použít více vláken na stejném CPU.

Tyto dva přístupy se vzájemně nevylučují a lze je kombinovat pro ještě vyšší výkon. Vyžadují však odlišné programovací modely a nástroje.

# Multithreading

Multithreading se snaží využít plný potenciál jednoho CPU.
Smysl je v tom, že běžné aplikace často nevyužívají CPU na plnou kapacitu a vznikají nečinné cykly, které lze využít pro další výpočty.
Použitím více vláken můžeme CPU udržet vytížené a dosáhnout lepšího výkonu.
CPU musí multithreading podporovat, což je u moderních procesorů běžné.
Hlavní výzvou je navrhnout program tak, aby byl rozdělen na nezávislé úlohy, které lze provádět paralelně bez vzájemného ovlivňování.
To často vyžaduje pečlivé zvážení datových závislostí a synchronizace mezi vlákny.
Multithreading v tomto kurzu probírat nebudeme, ale základy najdete v [poznámkách k přednášce](https://raw.githubusercontent.com/PavelStransky/PCInPhysics/main/Poznamky.pdf) Pavla Stránského.

# Multiprocessing

Použití více CPU je užitečné tehdy, když je možné nahradit jeden dlouhý běh programu několika kratšími běhy téhož programu.
Typické situace zahrnují:

- MC integraci, kdy se stejný program spouští vícekrát (jen s různými náhodnými semeny) pro lepší statistiku.

- Zpracování velkých dat, kdy lze data rozdělit na menší části, které lze zpracovat nezávisle.

V těchto případech můžeme použít cluster počítačů pro paralelní běh více instancí programu a na konci výsledky spojit.

# Terminologie

- **Node**: Celý server v clusteru.

- **CPU**: Central Processing Unit, výpočetní čip na nodu.

- **Core**: Jedna výpočetní jednotka v rámci CPU. Moderní CPU mají více jader, která mohou vykonávat instrukce nezávisle. V zásadě je pro každý běžící proces potřeba jedno jádro.


# Cluster Chimera

Cluster Chimera je výkonný výpočetní cluster dostupný studentům a výzkumníkům na Matematicko-fyzikální fakultě Univerzity Karlovy.
Je největší částí fakultního Metacentra (často označovaného jen jako HPC – High Performance Computing), které zahrnuje i další menší clustery a GPU cluster.

Podrobné informace o HPC clusteru najdete na [HPC GitLab stránce](https://gitlab.mff.cuni.cz/mff/hpc/clusters) nebo na jeho [webové stránce](https://www.mff.cuni.cz/en/hpc-cluster/general-information).
Skvělý [praktický průvodce](https://ipnp.cz/?page_id=8244), jak používat cluster Chimera, napsal Daniel Scheirich pro potřeby výzkumníků a studentů IPNP (Ústav částicové a jaderné fyziky), ale je to dobrý výchozí bod pro každého.


# SSH

Standardní nástroj pro připojení ke vzdálenému serveru je SSH (Secure Shell).
Umožňuje přihlásit se na server a spouštět příkazy, jako byste seděli přímo u něj.
SSH můžete použít také pro přenos souborů mezi lokálním počítačem a serverem.
Pro připojení ke clusteru Chimera použijte ve svém PowerShellu následující příkaz:
```powershell
ssh <username>@hpc.troja.mff.cuni.cz
```
Nahraďte `<username>` svým skutečným uživatelským jménem na clusteru (stejné jako pro systém SIS).
Budete vyzváni k zadání hesla a poté se přihlásíte do clusteru.

# Konfigurační soubor SSH

Abyste nemuseli pokaždé psát `<username>@hpc.troja.mff.cuni.cz`, můžete si vytvořit SSH konfigurační soubor se zkratkou připojení. Pak stačí napsat např. `ssh hpc`. Stačí vytvořit složku `C:\Users\<your_windows_username>\.ssh\` a v ní soubor `config` s následujícím obsahem:
```
Host hpc
    HostName hpc.troja.mff.cuni.cz
    User <username>
```
Nahraďte `<username>` svým skutečným uživatelským jménem na clusteru. Poté se do clusteru připojíte jednoduše příkazem `ssh hpc` v PowerShellu.

# SSH autentizace pomocí klíčů

Pokud nechcete při každém připojení ke clusteru zadávat heslo, můžete nastavit autentizaci pomocí SSH klíčů. To znamená vygenerovat na lokálním počítači dvojici klíčů (soukromý a veřejný) a veřejný klíč zkopírovat na cluster. Poté se budete ověřovat soukromým klíčem místo hesla.

Pro vygenerování dvojice SSH klíčů použijte v PowerShellu:
```powershell
ssh-keygen -t ed25519 -C "<your_email@example.com>"
```
Tím vznikne soukromý klíč (obvykle `id_ed25519`) a veřejný klíč (obvykle `id_ed25519.pub`) v adresáři `C:\Users\<your_windows_username>\.ssh\`.
Při výzvě k zadání souboru pro uložení klíče můžete stisknout Enter a přijmout výchozí umístění. Pro vyšší bezpečnost můžete nastavit i heslo (passphrase), nebo jej nechat prázdné.
Své soukromé klíče chraňte heslem!

Dále je potřeba zkopírovat veřejný klíč na cluster. Pokud na clusteru neexistuje soubor `.ssh/authorized_keys`, můžete použít:
```powershell
type $HOME\.ssh\id_ed25519.pub | ssh hpc "mkdir -p .ssh && tee .ssh/authorized_keys"
```
Jiná možnost je zkopírovat celý obsah souboru `id_ed25519.pub` z lokálního počítače na konec souboru `~/.ssh/authorized_keys` na vzdáleném stroji.

# SSH agent pro bezpečné držení soukromého klíče

Posunuli jsme se od nutnosti zadávat při každém přihlášení do Chimery heslo k HPC k nutnosti zadávat při každém přihlášení heslo k soukromému klíči.
To není moc efektivní :)
Naštěstí existuje nástroj, který drží váš soukromý klíč připravený, ale bezpečný, po celou dobu běhu lokální relace.
Ve Windows použijte službu SSH Agent (na jiných platformách něco obdobného).
Pro spuštění agenta udělejte:

1. Otevřete PowerShell jako správce (pravé tlačítko Start > Terminal (Admin)).

2. Spusťte následující příkazy pro start služby a nastavení automatického spouštění:
```powershell
Set-Service -Name ssh-agent -StartupType Automatic
Start-Service ssh-agent
```

Pokud chcete, aby si agent klíč pamatoval, spusťte:
```powershell
ssh-add $env:USERPROFILE\.ssh\id_ed25519
```


# Úprava souborů na clusteru pomocí VS Code

Nainstalujte rozšíření [Remote - SSH extension](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-ssh), abyste mohli upravovat soubory na clusteru ve VS Code. Po instalaci se můžete připojit kliknutím na ikonu se dvěma šipkami vlevo dole ve VS Code a volbou „Connect to Host“. Pak vyberte host, který jste definovali v SSH configu (např. `hpc`), a VS Code naváže spojení s clusterem. Poté můžete otevírat soubory a složky na clusteru přímo z VS Code a upravovat je, jako by byly lokálně.

> DŮLEŽITÉ: Nepoužívejte terminál ve VS Code ke spouštění programů na clusteru! Terminálová relace běží na login nodu, který není určen pro spouštění programů – sdílí ho všichni uživatelé a má omezené prostředky. Pro více informací, jak správně spouštět programy na clusteru, čtěte text níže.


# Slurm

Uživatelské úlohy na clusteru Chimera spravuje plánovač úloh Slurm. Je zodpovědný za přidělování prostředků (CPU, paměť atd.) uživatelským úlohám a za plánování jejich běhu na clusteru. Chcete-li na clusteru spustit program, musíte úlohu odeslat do Slurmu. Slurm ji pak naplánuje ke spuštění, jakmile jsou dostupné požadované prostředky. Existují dva základní typy úloh: interaktivní úlohy a dávkové úlohy.

# Interaktivní úlohy

Jde o úlohy, které běží v interaktivní relaci na clusteru. Když používáte JupyterHub, běžíte v interaktivní relaci na clusteru. Další způsob je použít příkaz `salloc` v terminálu na login nodu clusteru. Tím si rezervujete požadované prostředky a získáte shell na výpočetním nodu, kde můžete program spouštět interaktivně. Příklad spuštění interaktivní relace s 1 jádrem a 1 GB paměti na 24 hodin v partition `ffa`:
```bash
salloc -p ffa --cpus-per-task 1 --mem 1G --time=24:00:00
```

Všimněte si, že v partition `ffa` může být potřeba počkat na dostupnost prostředků. Partition s vysokou prioritou je `edu`; bohužel má krátký časový limit (4 hodiny):
```bash
salloc -p edu --cpus-per-task 1 --mem 1G --time=4:00:00
```


# Dávkové úlohy

Jsou vhodné pro programy, které nevyžadují interakci uživatele, zatímco interaktivní úlohy jsou užitečné pro ladění a testování kódu před odesláním jako dávkové úlohy.
Hlavní výpočetní práce na clusteru běží právě v dávkových úlohách.
Je potřeba použít příkaz `sbatch` v kombinaci s job skriptem. Job skript obsahuje příkazy pro spuštění programu i Slurm direktivy určující požadované prostředky. Příklad job skriptu, který spustí jednoduchý `echo`, s 1 jádrem, 100 MB paměti a časem 1 minuta v partition `ffa-preempt`:
```bash
#!/bin/bash
#SBATCH -p ffa-preempt
#SBATCH --cpus-per-task 1
#SBATCH --mem 100M
#SBATCH --time 00:01:00
#SBATCH --job-name test

# Here, do the real work
# Just a simple example:
echo "Hello from the cluster!"
```

První řádek `#!/bin/bash` je důležitý, protože říká systému, že jde o bash skript. Řádky začínající `#SBATCH` jsou Slurm direktivy určující požadované prostředky. Zbytek skriptu obsahuje příkazy pro spuštění programu.

> Tip: Vytvořte si na clusteru složku `test` a dejte skript tam. Pojmenujte ho libovolně, např. `job.sh`. Poté úlohu odešlete příkazem `sbatch job.sh` z adresáře `test`. Log soubory se pak vytvoří ve stejné složce a po dokončení je snadno zkontrolujete. Textový výstup úlohy bude v souboru `slurm-<job_id>.out`, kde `<job_id>` je ID přidělené úloze systémem Slurm.

# Slurm direktivy

- `#SBATCH -p <partition>` - určuje partition, ve které se úloha spustí. `ffa` je dobrá volba pro většinu úloh, ale fronta může být dlouhá. `edu` má velmi krátký časový limit (4 hodiny) a jen jednu úlohu na uživatele, ale má vyšší prioritu. `ffa-preempt` je dobrá volba pro krátké úlohy, které mohou být přerušeny jinými úlohami.

- `#SBATCH --cpus-per-task <num>` - určuje počet jader CPU požadovaných pro úlohu.

- `#SBATCH --mem <amount>` - určuje množství paměti požadované pro úlohu.

- `#SBATCH --time <time>` - určuje maximální čas běhu úlohy. Formát je `DD-HH:MM:SS`.

- `#SBATCH --job-name <name>` - určuje název úlohy, užitečný pro její identifikaci ve frontě.

- `#SBATCH --output <file>` - určuje soubor, do kterého se zapíše standardní výstup úlohy. Ve výchozím stavu je to `slurm-<job_id>.out`.

- `#SBATCH --error <file>` - určuje soubor, do kterého se zapíše standardní chybový výstup úlohy.

# Slurm příkazy

- `squeue` - zobrazí seznam aktuálně běžících a čekajících úloh. Volba `--me` zobrazí jen vaše úlohy.

- `scancel` - zruší běžící nebo čekající úlohu. Můžete zadat ID úlohy nebo použít `-u <username>` pro zrušení všech vašich úloh.

- `sinfo` - zobrazí informace o clusteru včetně dostupných partition a jejich stavu.


# Pole úloh (job arrays)

Pokud potřebujete spustit stejný program vícekrát s různými parametry (např. různými random seedy), můžete použít pole úloh. Pole úloh umožní odeslat více úloh jediným příkazem a každá úloha v poli má unikátní index, který lze použít k předání různých parametrů.
Pro vytvoření pole úloh použijte volbu `--array`. Unikátní index každé úlohy v poli je dostupný přes proměnnou prostředí `SLURM_ARRAY_TASK_ID` (a placeholder `%a` v názvech výstupních a chybových souborů).
Ukázkový job skript:
```bash
#!/bin/bash
#SBATCH --array 1-10
#SBATCH -p ffa-preempt
#SBATCH --cpus-per-task 1
#SBATCH --mem 100M
#SBATCH --time 00:01:00
#SBATCH --job-name test
#SBATCH --output slurm-%a.out
#SBATCH --error slurm-%a.err

# Here, do the real work
# Just a simple example:
echo "Hello from the cluster! Task ID: $SLURM_ARRAY_TASK_ID"
```


# Spuštění Python programu v dávkové úloze

Pro testování vytvořte jednoduchý Python skript `generate.py`. Skript by měl jako argument příkazové řádky přijmout index úlohy v poli a vygenerovat náhodná čísla, přičemž index použije jako random seed. Například:
```python
import numpy as np
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("-i", "--index", type = int)
args = parser.parse_args()

np.random.seed(args.index)
a = np.random.normal(size = 1000000)

print(f"Generated {len(a)} random numbers with seed {args.index}")
print(f"Mean: {np.mean(a)}, Std: {np.std(a)}")
```

Skript potřebuje `numpy`, takže na clusteru použijte vhodné Python prostředí. Následující venv je pro nás dostačující:
```bash
/singularity/ucjf/venv_praktikum/bin/activate
```

> DŮLEŽITÉ: Při vytváření vlastního venv se ujistěte, že jste v interaktivní relaci, ne na login nodu!

Nakonec vytvořte job skript `job.sh`, který spustí Python skript v poli úloh:
```bash
#!/bin/bash
#SBATCH --array 1-10
#SBATCH -p ffa-preempt
#SBATCH --cpus-per-task 1
#SBATCH --mem 1G
#SBATCH --time 00:01:00
#SBATCH --job-name test
#SBATCH --output slurm-%a.out
#SBATCH --error slurm-%a.err

# Activate the Python virtual environment
source /singularity/ucjf/venv_praktikum/bin/activate

# Run the Python script with the index of the job in the array as an argument
python generate.py -i $SLURM_ARRAY_TASK_ID
```

Úlohu odešlete příkazem `sbatch job.sh`.


# Generování submission souborů z Python skriptu

Často potřebujete spustit velké množství úloh s různými parametry a vytvářet pro každou úlohu samostatný job skript může být únavné. V takových případech může být užitečné job skripty generovat pomocí Python skriptu. Snadno tak vytvoříte velké množství job skriptů s různými parametry a pak je odešlete najednou. Často také budete potřebovat pomocí Python skriptu vygenerovat hlavní Python skript (který budou job skripty spouštět).
